# Experimento anterior: reranker seletivo

Decide **se aplica BGE**, usando os resultados da comparação BM25/BGE.
É distinto do TCC, que decide **se expande a consulta** antes do reranqueamento.
Não carrega corpus ou BGE. Lê os CSVs existentes; o benchmark query-only usa um
modelo de embeddings, que pode precisar ser baixado.

Mantenha as três bases e amostras suficientes para as validações estratificada e
leave-one-dataset-out do experimento original.


In [ ]:
from pathlib import Path
import sys

# Local: detecta o projeto. Colab: ajuste para a pasta do repositório no Drive.
# Antes, monte o Drive em uma célula: from google.colab import drive; drive.mount('/content/drive')
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src" / "qexp").is_dir()),
                    Path("/content/drive/MyDrive/PLN"))
if not (PROJECT_ROOT / "src" / "qexp").is_dir():
    raise FileNotFoundError("Ajuste PROJECT_ROOT para a pasta do projeto, contendo src/ e configs/.")
PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Projeto:", PROJECT_ROOT)


In [ ]:
# Execute uma vez no ambiente/Colab. O pacote precisa estar disponível junto ao notebook.
%pip install -e "{PROJECT_ROOT}[notebooks]"


In [ ]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from pln_legacy.io import load_baseline_results
from pln_legacy.selective import *

# Pode apontar para a pasta de uma nova execução do notebook anterior.
BASELINE_RESULTS = PROJECT_ROOT / "PLN_results_outputs"
OUTPUT_DIR = PROJECT_ROOT / "results" / "reranker_seletivo" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
rankings_all, results_all = load_baseline_results(BASELINE_RESULTS)
query_deltas = build_query_deltas(results_all)
# Valida os requisitos do protocolo antes de treinar.
labels = build_classifier_dataset(rankings_all, query_deltas)
if labels["dataset"].nunique() < 2 or labels["rerank_beneficial"].nunique() < 2 or labels["rerank_beneficial"].value_counts().min() < 5:
    raise ValueError("Use múltiplos datasets e pelo menos 5 queries em cada classe para o protocolo original.")
for dataset in labels["dataset"].unique():
    if labels.loc[labels["dataset"] != dataset, "rerank_beneficial"].nunique() < 2:
        raise ValueError(f"O fold sem {dataset} precisa de exemplos das duas classes.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(OUTPUT_DIR)
print("Saídas:", OUTPUT_DIR)


## Classificador: quando aplicar o reranker?

Esta seção treina uma política seletiva para prever se o BGE-Reranker melhora uma query. O rótulo é `delta_nDCG@10 > 0`; empate e piora são tratados como **não rerankear**.

As features usam somente informações disponíveis antes do rerankeamento: texto da query, distribuição dos scores BM25, margens entre candidatos e cobertura lexical nos documentos recuperados. Qrels e métricas pós-rerankeamento são usados somente para criar o rótulo.

As funções estão em `src/pln_legacy/selective.py`.


In [ ]:
query_classifier_results = run_query_classifier_experiment(
    rankings=rankings_all,
    metric_deltas=query_deltas,
    output_dir=".",
)

display(query_classifier_results["summary"].round(4))
display(
    query_classifier_results["dataset"]
    .groupby("dataset")["rerank_beneficial"]
    .agg(queries="size", beneficial_rate="mean")
    .reset_index()
    .round(4)
)


## Benchmark query-only

### Comparação de modelos para decidir quando rerankear

Os modelos abaixo recebem **somente o texto da query**. Scores, posições e documentos do BM25 não são usados como features. O rótulo continua sendo `delta_nDCG@10 > 0`.

São comparados TF-IDF de palavras, TF-IDF de caracteres, *gradient boosting* com atributos linguísticos, embedding com regressão logística e embedding com uma pequena rede neural MLP.

Esta análise é exploratória. A validação estratificada mede generalização entre queries misturando os três datasets, enquanto *leave-one-dataset-out* é um cenário mais rigoroso de transferência para um domínio não visto.


In [ ]:
query_only_benchmark = run_query_only_benchmark(
    rankings=rankings_all,
    metric_deltas=query_deltas,
    output_dir=".",
)

query_only_summary = query_only_benchmark["summary"].sort_values(
    ["evaluation", "roc_auc"], ascending=[True, False]
)
display(query_only_summary.round(4))

plt.figure(figsize=(11, 6))
sns.barplot(
    data=query_only_summary,
    x="roc_auc", y="model", hue="evaluation",
)
plt.axvline(0.5, color="black", linestyle="--", linewidth=1)
plt.xlim(0, 1)
plt.title("Benchmark query-only para decidir quando rerankear")
plt.tight_layout()
plt.savefig("query_only_model_benchmark.png", dpi=160, bbox_inches="tight")
plt.show()

query_only_training_history = query_only_benchmark["training_history"]
display(query_only_training_history.head())

if not query_only_training_history.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)

    sns.lineplot(
        data=query_only_training_history,
        x="epoch", y="loss", hue="evaluation",
        estimator="mean", ci="sd", ax=axes[0],
    )
    axes[0].set_title("Query-only MLP - loss por epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    validation_history = query_only_training_history.dropna(subset=["validation_score"])
    sns.lineplot(
        data=validation_history,
        x="epoch", y="validation_score", hue="evaluation",
        estimator="mean", ci="sd", ax=axes[1],
    )
    axes[1].set_title("Query-only MLP - validação por epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy de validação")

    plt.tight_layout()
    plt.savefig("query_only_mlp_training_epochs.png", dpi=160, bbox_inches="tight")
    plt.show()


## Auditoria prática da política query-only

Esta seção usa as predições fora-do-fold do benchmark query-only para simular uma política seletiva: executar sempre o BM25, chamar o BGE-Reranker apenas quando o classificador prediz benefício e comparar qualidade/custo contra sempre rerankear.

O tempo seletivo é estimado como `tempo_BM25 + decisão * (tempo_BM25+BGE - tempo_BM25) + tempo_inferência_classificador`. Para modelos baseados em embedding, o tempo médio de gerar embeddings das queries também entra na inferência do classificador.


In [ ]:
query_only_policy_audit, query_only_inference_timing = build_query_only_policy_audit(
    query_only_benchmark,
    results_all,
)
query_only_policy_summary = summarize_query_only_policy(query_only_policy_audit)

query_only_policy_audit.to_csv("query_only_policy_query_audit.csv", index=False)
query_only_policy_summary.to_csv("query_only_policy_time_quality_summary.csv", index=False)
query_only_inference_timing.to_csv("query_only_inference_timing.csv", index=False)

display(query_only_policy_summary.sort_values(
    ["evaluation", "mean_delta_selective_policy"], ascending=[True, False]
).round(6))
display(query_only_inference_timing.round(6))

example_model = "embedding_mlp"
example_evaluation = "leave_one_dataset_out"
query_only_policy_examples = query_only_policy_audit[
    (query_only_policy_audit["model"] == example_model)
    & (query_only_policy_audit["evaluation"] == example_evaluation)
].copy()
if query_only_policy_examples.empty:
    query_only_policy_examples = query_only_policy_audit.copy()

query_only_policy_examples["confidence"] = (query_only_policy_examples["rerank_probability"] - 0.5).abs()
query_only_policy_examples = query_only_policy_examples.sort_values(
    ["correct_decision", "confidence"], ascending=[True, False]
).head(20)

display(query_only_policy_examples[[
    "dataset", "query_id", "query_text", "rerank_probability", "decision",
    "actual_outcome", "delta_nDCG@10", "correct_decision",
    "bm25_elapsed_sec", "bm25_bge_elapsed_sec", "estimated_selective_elapsed_sec",
    "time_saved_vs_always_rerank_sec",
]].round(6))

plt.figure(figsize=(11, 5))
sns.barplot(
    data=query_only_policy_summary,
    x="mean_time_selective_policy_sec", y="model", hue="evaluation",
)
plt.axvline(
    query_only_policy_audit["bm25_bge_elapsed_sec"].mean(),
    color="black", linestyle="--", linewidth=1, label="Sempre rerankear",
)
plt.title("Tempo médio estimado da política query-only")
plt.xlabel("Tempo médio por query (s)")
plt.ylabel("Modelo")
plt.tight_layout()
plt.savefig("query_only_policy_time_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=query_only_policy_summary,
    x="mean_time_selective_policy_sec",
    y="mean_delta_selective_policy",
    hue="evaluation",
    style="model",
    s=100,
)
plt.axhline(query_only_policy_audit["delta_nDCG@10"].mean(), color="black", linestyle="--", linewidth=1)
plt.title("Trade-off qualidade vs tempo da política query-only")
plt.xlabel("Tempo médio estimado por query (s)")
plt.ylabel("Delta nDCG@10 médio seletivo")
plt.tight_layout()
plt.savefig("query_only_policy_quality_time_tradeoff.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(archive)
